# **WEEK 5 — Multivariate Time Series: VAR, Cointegration, and GARCH**

>Jerald B. Bongalos | PhD in Data Science | Asian Institute of Management

---

*Reference:*
>Shumway, R. H., & Stoffer, D. S. *Time Series Analysis and Its Applications*
>(Ch 5.1–5.6: Additional Time Domain Topics)

---

**Purpose of this Notebook**

This notebook extends time series analysis from the **univariate** setting (Weeks 1–4)
to the **multivariate** and **conditional variance** settings. We move from modeling
a single series to modeling systems of interacting series and volatility dynamics.

**The goal is to understand:**

- how VAR models capture joint dynamics of multiple series,
- what Granger causality means (and does not mean),
- why spurious regression arises and how cointegration resolves it,
- how ARCH/GARCH models capture volatility clustering,
- and how these tools connect to the univariate foundations.

**Learning Outcomes**

By the end of this notebook, you should be able to:

1. Formulate and estimate VAR($p$) models
2. Compute and interpret impulse response functions (IRFs)
3. Test and interpret Granger causality
4. Explain spurious regression and the need for cointegration testing
5. Distinguish between the Engle–Granger and Johansen approaches
6. Formulate and estimate VECM when cointegration is present
7. Detect volatility clustering and fit ARCH/GARCH models
8. Explain why GARCH residuals are white noise but not independent

---

**Notebook Structure**

| Part | Topic | Type | Priority |
|------|-------|------|----------|
| 1 | VAR Models: Joint Dynamics | Theory + Simulation | CRITICAL |
| 2 | Impulse Response Functions | Theory + Simulation | CRITICAL |
| 3 | Granger Causality | Theory + Simulation | CRITICAL |
| 4 | Spurious Regression and Unit Roots | Theory + Simulation | CRITICAL |
| 5 | Cointegration and VECM | Theory + Simulation | CRITICAL |
| 6 | ARCH/GARCH: Volatility Modeling | Theory + Simulation | HIGH |
| 7 | Synthesis (Exam-Ready) | Summary | — |
| 8 | Self-Test Questions | Exam Preparation | — |


In [ ]:
# ============================================================
# SETUP
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, grangercausalitytests, acf
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.irf import IRAnalysis
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import norm

np.random.seed(123)

# --- Shared: simulate VAR(1) bivariate ---
def simulate_var1(A, n=500, sigma=None, burnin=200):
    """Simulate VAR(1): X_t = A X_{t-1} + eps_t."""
    k = A.shape[0]
    if sigma is None:
        sigma = np.eye(k)
    N = n + burnin
    eps = np.random.multivariate_normal(np.zeros(k), sigma, N)
    X = np.zeros((N, k))
    for t in range(1, N):
        X[t] = A @ X[t-1] + eps[t]
    return X[burnin:]

print("Setup complete.")


## **PART 1 — VAR Models: Joint Dynamics**

> **Why this part matters**
>
> Univariate ARMA models capture dynamics of a single series.
> In practice, multiple series often **interact** — GDP affects unemployment,
> temperature affects PM2.5, one pollutant affects another.
>
> The **Vector Autoregressive (VAR)** model is the multivariate extension of AR($p$).

---

### **1.1 The VAR($p$) Model**

For a $k$-dimensional time series $\mathbf{X}_t = (X_{1t}, X_{2t}, \ldots, X_{kt})'$:

$$
\mathbf{X}_t = \mathbf{A}_1 \mathbf{X}_{t-1} + \mathbf{A}_2 \mathbf{X}_{t-2} + \cdots + \mathbf{A}_p \mathbf{X}_{t-p} + \boldsymbol{\varepsilon}_t
$$

where:
- $\mathbf{A}_j$ are $k \times k$ coefficient matrices
- $\boldsymbol{\varepsilon}_t \sim \mathrm{WN}(\mathbf{0}, \boldsymbol{\Sigma})$ is $k$-dimensional white noise
- $\boldsymbol{\Sigma}$ is the $k \times k$ innovation covariance matrix

---

### **1.2 VAR(1) Bivariate Example**

For $k = 2$:

$$
\begin{pmatrix} X_{1t} \\ X_{2t} \end{pmatrix} = \begin{pmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{pmatrix} \begin{pmatrix} X_{1,t-1} \\ X_{2,t-1} \end{pmatrix} + \begin{pmatrix} \varepsilon_{1t} \\ \varepsilon_{2t} \end{pmatrix}
$$

- $a_{12} \neq 0$: lagged $X_2$ affects current $X_1$ (cross-dependence)
- $a_{21} \neq 0$: lagged $X_1$ affects current $X_2$
- Off-diagonal entries capture **lead-lag relationships**

---

### **1.3 Stationarity**

VAR($p$) is stable (stationary) if all eigenvalues of the companion matrix lie **inside the unit circle**.

For VAR(1): all eigenvalues of $\mathbf{A}$ must satisfy $|\lambda_i| < 1$.

---

### **1.4 Estimation**

Each equation of a VAR can be estimated by **OLS** (equation by equation).
This is efficient when all equations have the same regressors.

**Lag order selection**: use AIC, BIC, or HQ applied to the VAR system.

> **Exam-safe statement:** "VAR($p$) is the multivariate extension of AR($p$). It captures
> both own-lag and cross-lag dependence. Estimation is by equation-by-equation OLS."


In [ ]:
# ============================================================
# PART 1: VAR(1) Simulation and Estimation
# ============================================================

np.random.seed(123)

# --- VAR(1) with cross-dependence ---
A = np.array([[0.7, 0.2],
              [-0.1, 0.5]])

Sigma = np.array([[1.0, 0.3],
                   [0.3, 1.0]])

X = simulate_var1(A, n=500, sigma=Sigma)
df = pd.DataFrame(X, columns=["$X_1$", "$X_2$"])

# --- Time plots ---
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(df["$X_1$"], linewidth=0.8, color="#2c3e50")
axes[0].set_title("$X_{1t}$", fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[1].plot(df["$X_2$"], linewidth=0.8, color="#e74c3c")
axes[1].set_title("$X_{2t}$", fontsize=12)
axes[1].set_xlabel("$t$")
axes[1].grid(True, alpha=0.3)
plt.suptitle("Simulated VAR(1): $A_{12}=0.2$, $A_{21}=-0.1$", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# --- Fit VAR ---
df_fit = pd.DataFrame(X, columns=["X1", "X2"])
model = VAR(df_fit)

# Lag order selection
lag_order = model.select_order(maxlags=8)
print("=== LAG ORDER SELECTION ===")
print(lag_order.summary())

# Fit VAR(1)
results = model.fit(1)
print("\n=== VAR(1) ESTIMATION ===")
print(results.summary())

# --- Compare estimated vs true coefficients ---
A_hat = results.coefs[0]
print("\n=== COEFFICIENT COMPARISON ===")
print(f"True A:\n{A}")
print(f"\nEstimated A:\n{np.round(A_hat, 4)}")


## **PART 2 — Impulse Response Functions**

> **Why this part matters**
>
> An IRF traces the dynamic effect of a **one-time shock** to one variable
> on the entire system over time. It is the multivariate analog of the
> $\psi$-weights from the Wold representation.

---

### **2.1 MA($\infty$) Representation of VAR($p$)**

A stable VAR($p$) can be written as:

$$
\mathbf{X}_t = \sum_{j=0}^{\infty} \boldsymbol{\Psi}_j\, \boldsymbol{\varepsilon}_{t-j}
$$

where $\boldsymbol{\Psi}_0 = \mathbf{I}_k$ and the $\boldsymbol{\Psi}_j$ matrices are the **multivariate $\psi$-weights**.

For VAR(1): $\boldsymbol{\Psi}_j = \mathbf{A}^j$.

---

### **2.2 Interpretation**

The $(i,j)$ element of $\boldsymbol{\Psi}_h$:

$$
[\boldsymbol{\Psi}_h]_{ij} = \frac{\partial X_{i,t+h}}{\partial \varepsilon_{j,t}}
$$

This is the **response of variable $i$** at horizon $h$ to a **unit shock in variable $j$** at time $t$.

---

### **2.3 Orthogonalized IRFs**

If the innovation covariance $\boldsymbol{\Sigma}$ is not diagonal (shocks are correlated),
we orthogonalize using the **Cholesky decomposition**: $\boldsymbol{\Sigma} = \mathbf{P}\mathbf{P}'$.

Orthogonalized IRFs: $\boldsymbol{\Theta}_h = \boldsymbol{\Psi}_h \mathbf{P}$

> **Caution:** Cholesky ordering matters. The first variable's shock is kept as-is;
> the second's is orthogonalized to the first, etc. The ordering encodes **contemporaneous causality assumptions**.


In [ ]:
# ============================================================
# PART 2: Impulse Response Functions
# ============================================================

irf = results.irf(20)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for i in range(2):
    for j in range(2):
        irvals = irf.irfs[:, i, j]
        axes[i, j].plot(irvals, linewidth=2, color="#2c3e50", marker="o", markersize=3)
        axes[i, j].axhline(0, color="gray", linestyle="--", alpha=0.5)
        axes[i, j].set_title(f"Response of $X_{i+1}$ to shock in $X_{j+1}$", fontsize=11)
        axes[i, j].set_xlabel("Horizon $h$")
        axes[i, j].set_ylabel("Response")
        axes[i, j].grid(True, alpha=0.3)

plt.suptitle("Impulse Response Functions — VAR(1)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# --- Orthogonalized IRFs ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

orth_irfs = irf.orth_irfs
for i in range(2):
    for j in range(2):
        irvals = orth_irfs[:, i, j]
        lo = irf.orth_lrs[:, i, j] if hasattr(irf, 'orth_lrs') else irvals  # fallback
        hi = irf.orth_urs[:, i, j] if hasattr(irf, 'orth_urs') else irvals
        axes[i, j].plot(irvals, linewidth=2, color="#e74c3c", marker="o", markersize=3)
        axes[i, j].axhline(0, color="gray", linestyle="--", alpha=0.5)
        axes[i, j].set_title(f"Orth. response of $X_{i+1}$ to shock in $X_{j+1}$", fontsize=11)
        axes[i, j].set_xlabel("Horizon $h$")
        axes[i, j].grid(True, alpha=0.3)

plt.suptitle("Orthogonalized IRFs (Cholesky: $X_1$ ordered first)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - Diagonal IRFs (X1->X1, X2->X2): own-shock persistence (decaying)")
print("  - Off-diagonal IRFs: cross-variable transmission")
print("  - A12=0.2 means X2 shock transmits positively to X1")
print("  - A21=-0.1 means X1 shock transmits negatively to X2")
print("  - All responses decay to 0 (stable VAR)")


## **PART 3 — Granger Causality**

> **Why this part matters**
>
> Granger causality asks: *does the past of $X_2$ help predict $X_1$
> beyond the past of $X_1$ alone?* It is **predictive causality**,
> not structural or physical causality.

---

### **3.1 Definition**

$X_2$ **Granger-causes** $X_1$ if:

$$
\mathrm{Var}\!\left(X_{1,t+1} \mid X_{1,t}, X_{1,t-1}, \ldots\right) > \mathrm{Var}\!\left(X_{1,t+1} \mid X_{1,t}, X_{1,t-1}, \ldots, X_{2,t}, X_{2,t-1}, \ldots\right)
$$

In words: including the history of $X_2$ **reduces** the forecast error variance of $X_1$.

---

### **3.2 Testing in a VAR Framework**

In VAR(1) for $(X_1, X_2)$:

$$
X_{1t} = a_{11} X_{1,t-1} + a_{12} X_{2,t-1} + \varepsilon_{1t}
$$

$X_2$ does **not** Granger-cause $X_1$ $\iff$ $a_{12} = 0$ (all cross-lag coefficients are zero).

The test is a standard **F-test** (or Wald/chi-squared test) on the joint restriction that all coefficients from $X_2$'s lags in $X_1$'s equation are zero.

---

### **3.3 What Granger Causality Does NOT Mean**

- It does **not** imply physical or structural causation
- It can arise from a **common cause** (confounding)
- It is **direction-dependent**: $X_2 \to X_1$ does not imply $X_1 \to X_2$
- It depends on the **information set** (adding a third variable can change results)
- It assumes **stationarity** — testing on nonstationary data gives spurious results

> **Exam-critical:** "Granger causality is predictive causality — it tests whether past values of one series
> improve forecasts of another. It does not establish structural or physical causation."


In [ ]:
# ============================================================
# PART 3: Granger Causality Testing
# ============================================================

np.random.seed(123)

# --- Case 1: Bidirectional Granger causality (A12 != 0 AND A21 != 0) ---
A_bidir = np.array([[0.7, 0.2],
                     [-0.1, 0.5]])
X_bidir = simulate_var1(A_bidir, n=500)
df_bidir = pd.DataFrame(X_bidir, columns=["X1", "X2"])

# --- Case 2: Unidirectional (X2 -> X1 only: A12 != 0, A21 = 0) ---
A_unidir = np.array([[0.7, 0.3],
                      [0.0, 0.5]])
X_unidir = simulate_var1(A_unidir, n=500)
df_unidir = pd.DataFrame(X_unidir, columns=["X1", "X2"])

# --- Case 3: No causality (diagonal A) ---
A_none = np.array([[0.7, 0.0],
                    [0.0, 0.5]])
X_none = simulate_var1(A_none, n=500)
df_none = pd.DataFrame(X_none, columns=["X1", "X2"])

# --- Test all cases ---
cases = [
    ("Bidirectional (A12=0.2, A21=-0.1)", df_bidir),
    ("Unidirectional (A12=0.3, A21=0)", df_unidir),
    ("No causality (diagonal A)", df_none),
]

print("=" * 70)
print("GRANGER CAUSALITY TESTS (lag=1, alpha=0.05)")
print("=" * 70)

for label, df_case in cases:
    print(f"\n--- {label} ---")
    for direction, cols in [("X2 -> X1", ["X1", "X2"]), ("X1 -> X2", ["X2", "X1"])]:
        try:
            result = grangercausalitytests(df_case[cols], maxlag=1, verbose=False)
            pval = result[1][0]["ssr_ftest"][1]
            verdict = "REJECT (Granger-causes)" if pval < 0.05 else "FAIL TO REJECT (no Granger causality)"
            print(f"  {direction}: F-test p = {pval:.4f} -> {verdict}")
        except Exception as e:
            print(f"  {direction}: ERROR ({e})")

print("\nExpected results:")
print("  Bidirectional: both directions significant")
print("  Unidirectional: X2->X1 significant, X1->X2 not")
print("  No causality: neither direction significant")


## **PART 4 — Spurious Regression and Unit Roots**

> **Why this part matters**
>
> Regressing one random walk on another gives **statistically significant** results
> even when the two series are completely independent. This is the **spurious regression** problem
> (Granger and Newbold, 1974).

---

### **4.1 The Problem**

If $X_t$ and $Y_t$ are independent random walks:

$$
X_t = X_{t-1} + \varepsilon_t, \qquad Y_t = Y_{t-1} + \eta_t, \qquad \varepsilon_t \perp \eta_t
$$

Regressing $Y_t$ on $X_t$ typically yields:
- High $R^2$
- Significant $t$-statistics
- But the regression is **meaningless** — both series just wander

---

### **4.2 Why This Happens**

- OLS assumes stationarity (or at least stable moments)
- Random walks have **growing variance** and **persistent levels**
- Two trending series appear correlated simply because both drift
- Standard $t$-statistics have **nonstandard distributions** under nonstationarity

---

### **4.3 Detection**

Signs of spurious regression:
- Very high $R^2$ but **autocorrelated residuals**
- Regression residuals are **nonstationary** (fail ADF test)
- Durbin–Watson statistic near 0 (extreme positive autocorrelation)

> **Rule of thumb (Granger):** If $R^2 > DW$, suspect spurious regression.

This motivates two key tools:
1. **Unit root testing** (ADF) — before regression, check if series are $I(1)$
2. **Cointegration testing** — if both are $I(1)$, check if a linear combination is $I(0)$


In [ ]:
# ============================================================
# PART 4: Spurious Regression Demonstration
# ============================================================
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

np.random.seed(42)
n = 300

# --- Two INDEPENDENT random walks ---
x_rw = np.cumsum(np.random.normal(0, 1, n))
y_rw = np.cumsum(np.random.normal(0, 1, n))

# --- OLS: Y on X (spurious) ---
X_reg = add_constant(x_rw)
ols_result = OLS(y_rw, X_reg).fit()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Scatter
axes[0].scatter(x_rw, y_rw, s=5, alpha=0.5, color="#2c3e50")
axes[0].set_title(f"Scatter: $R^2 = {ols_result.rsquared:.3f}$ (SPURIOUS!)", fontsize=11)
axes[0].set_xlabel("$X_t$ (random walk)")
axes[0].set_ylabel("$Y_t$ (independent random walk)")
axes[0].grid(True, alpha=0.3)

# Time plot
axes[1].plot(x_rw, linewidth=1, label="$X_t$", color="#2c3e50")
axes[1].plot(y_rw, linewidth=1, label="$Y_t$", color="#e74c3c")
axes[1].set_title("Two Independent Random Walks", fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Residuals
resid = ols_result.resid
axes[2].plot(resid, linewidth=0.8, color="#8e44ad")
axes[2].set_title("OLS Residuals (should be stationary if real)", fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Diagnostics ---
print("=== SPURIOUS REGRESSION DIAGNOSTICS ===")
print(f"  R-squared:       {ols_result.rsquared:.4f}")
print(f"  t-stat on X:     {ols_result.tvalues[1]:.4f} (p = {ols_result.pvalues[1]:.4f})")
print(f"  Durbin-Watson:   {np.sum(np.diff(resid)**2) / np.sum(resid**2):.4f}")
print(f"  R^2 > DW?        {'YES -> SPURIOUS' if ols_result.rsquared > np.sum(np.diff(resid)**2)/np.sum(resid**2) else 'No'}")

# ADF on residuals
stat, pval, _, _, _, _ = adfuller(resid, autolag="AIC")
print(f"\n  ADF on residuals: stat={stat:.4f}, p={pval:.4f}")
print(f"  Residuals stationary? {'YES' if pval < 0.05 else 'NO -> confirms spurious'}")

print("\nConclusion:")
print("  High R^2 and significant t-stat, but residuals are nonstationary.")
print("  The series are INDEPENDENT — this is textbook spurious regression.")


## **PART 5 — Cointegration and VECM**

> **Why this part matters**
>
> Part 4 showed that regressing $I(1)$ series gives spurious results.
> But some $I(1)$ series are linked by a **long-run equilibrium** — they wander together.
> This is **cointegration** (Engle and Granger, 1987).

---

### **5.1 Definition**

Two $I(1)$ series $X_t$ and $Y_t$ are **cointegrated** if there exists $\beta$ such that:

$$
Z_t = Y_t - \beta X_t \sim I(0) \quad \text{(stationary)}
$$

$Z_t$ is called the **equilibrium error** — deviations from the long-run relationship are temporary.

---

### **5.2 Engle–Granger Two-Step Procedure**

1. **Step 1:** Regress $Y_t$ on $X_t$ by OLS: $\hat{\beta}$ is the cointegrating coefficient
2. **Step 2:** Test the residuals $\hat{Z}_t = Y_t - \hat{\beta}X_t$ for stationarity (ADF test with special critical values)

If $\hat{Z}_t$ is stationary → cointegration is present.

---

### **5.3 The VECM (Vector Error Correction Model)**

If $X_t$ and $Y_t$ are cointegrated, the correct model is **not** VAR in levels or VAR in differences, but the **VECM**:

$$
\Delta \mathbf{X}_t = \boldsymbol{\alpha}\boldsymbol{\beta}' \mathbf{X}_{t-1} + \sum_{j=1}^{p-1} \boldsymbol{\Gamma}_j \Delta\mathbf{X}_{t-j} + \boldsymbol{\varepsilon}_t
$$

where:
- $\boldsymbol{\beta}$: cointegrating vector(s) — the long-run relationship
- $\boldsymbol{\alpha}$: loading/adjustment vector — speed of error correction
- $\boldsymbol{\alpha}\boldsymbol{\beta}' \mathbf{X}_{t-1}$: the error correction term

---

### **5.4 Johansen Procedure**

For systems with $k > 2$ variables, the **Johansen test** determines the **number of cointegrating relationships** (rank $r$ of $\boldsymbol{\Pi} = \boldsymbol{\alpha}\boldsymbol{\beta}'$).

- $r = 0$: no cointegration (use VAR in differences)
- $0 < r < k$: cointegration present (use VECM with rank $r$)
- $r = k$: all series are stationary (use VAR in levels)

---

### **5.5 Key Distinctions**

| Scenario | Series | Model |
|---|---|---|
| Both stationary ($I(0)$) | $X_t, Y_t \sim I(0)$ | VAR in levels |
| Both $I(1)$, NOT cointegrated | $X_t, Y_t \sim I(1)$ | VAR in **differences** |
| Both $I(1)$, cointegrated | $Z_t = Y_t - \beta X_t \sim I(0)$ | **VECM** |
| Mixed orders | $X_t \sim I(0)$, $Y_t \sim I(1)$ | Cannot be cointegrated |

> **Exam-critical:** "Differencing cointegrated series throws away the long-run equilibrium information.
> VECM preserves both short-run dynamics and the long-run relationship."


In [ ]:
# ============================================================
# PART 5: Cointegration — Simulation and Engle-Granger Test
# ============================================================

np.random.seed(123)
n = 500

# --- Simulate cointegrated pair ---
# Common stochastic trend + equilibrium relationship
eps_x = np.random.normal(0, 1, n)
eps_y = np.random.normal(0, 0.8, n)

# X is a random walk
x_ci = np.cumsum(eps_x)

# Y shares the stochastic trend: Y = 1.5*X + stationary noise
beta_true = 1.5
z_stationary = np.zeros(n)
rho_z = 0.5  # mean-reverting equilibrium error
for t in range(1, n):
    z_stationary[t] = rho_z * z_stationary[t-1] + eps_y[t]

y_ci = beta_true * x_ci + z_stationary

# --- Plot cointegrated pair ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(x_ci, linewidth=1, label="$X_t$ (random walk)", color="#2c3e50")
axes[0].plot(y_ci, linewidth=1, label="$Y_t$ (cointegrated)", color="#e74c3c")
axes[0].set_title("Cointegrated Pair", fontsize=11)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Equilibrium error
z_hat = y_ci - beta_true * x_ci
axes[1].plot(z_hat, linewidth=0.8, color="#27ae60")
axes[1].axhline(0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_title("Equilibrium Error $Z_t = Y_t - 1.5 X_t$ (stationary)", fontsize=11)
axes[1].grid(True, alpha=0.3)

# Compare: independent random walks
y_indep = np.cumsum(np.random.normal(0, 1, n))
z_indep = y_indep - 1.5 * x_ci
axes[2].plot(z_indep, linewidth=0.8, color="#8e44ad")
axes[2].set_title("Independent RWs: $Y - 1.5X$ (nonstationary)", fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Engle-Granger procedure ---
print("=" * 60)
print("ENGLE-GRANGER COINTEGRATION TEST")
print("=" * 60)

# Step 1: OLS regression Y on X
X_reg = add_constant(x_ci)
eg_result = OLS(y_ci, X_reg).fit()
beta_hat = eg_result.params[1]
resid_eg = eg_result.resid
print(f"\nStep 1 - OLS: Y = {eg_result.params[0]:.4f} + {beta_hat:.4f} * X")
print(f"  True beta = {beta_true}")

# Step 2: ADF on residuals
stat, pval, _, _, crit, _ = adfuller(resid_eg, autolag="AIC")
print(f"\nStep 2 - ADF on residuals:")
print(f"  ADF stat = {stat:.4f}, p-value = {pval:.4f}")
print(f"  Critical values: {crit}")
if pval < 0.05:
    print(f"  Result: REJECT -> residuals are stationary -> COINTEGRATED")
else:
    print(f"  Result: FAIL TO REJECT -> no cointegration")

# --- Repeat for independent pair ---
print(f"\n{'=' * 60}")
print("CONTROL: Independent Random Walks")
print("=" * 60)
eg_indep = OLS(y_indep, X_reg).fit()
stat_i, pval_i, _, _, _, _ = adfuller(eg_indep.resid, autolag="AIC")
print(f"  ADF on residuals: stat={stat_i:.4f}, p={pval_i:.4f}")
print(f"  Result: {'REJECT (spurious!)' if pval_i < 0.05 else 'FAIL TO REJECT -> correctly detects no cointegration'}")

# --- Unit root tests on individual series ---
print(f"\n{'=' * 60}")
print("UNIT ROOT TESTS (Individual Series)")
print("=" * 60)
for label, s in [("X (random walk)", x_ci), ("Y (cointegrated)", y_ci), ("Z = Y-bX (equilibrium error)", z_hat)]:
    stat_s, pval_s, _, _, _, _ = adfuller(s, autolag="AIC")
    result = "I(0) stationary" if pval_s < 0.05 else "I(1) nonstationary"
    print(f"  {label}: ADF p = {pval_s:.4f} -> {result}")


## **PART 6 — ARCH/GARCH: Volatility Modeling**

> **Why this part matters**
>
> In Week 1 (Part 2), we showed that ARCH processes are **white noise** (uncorrelated)
> but **not independent** — their squared values are autocorrelated. ARCH/GARCH models
> formalize this: they model the **conditional variance** (volatility) of a process.
>
> This is essential for financial time series, where returns are approximately
> uncorrelated but exhibit **volatility clustering**.

---

### **6.1 ARCH(1) Model**

$$
X_t = \sigma_t \, z_t, \qquad z_t \sim \mathrm{iid}(0,1)
$$

$$
\sigma_t^2 = \alpha_0 + \alpha_1 X_{t-1}^2
$$

The conditional variance depends on the **previous squared observation**.

- $\alpha_0 > 0$: baseline variance
- $\alpha_1 \geq 0$: volatility persistence (large $X_{t-1}^2$ → large $\sigma_t^2$)
- Stationarity: $\alpha_1 < 1$

---

### **6.2 GARCH(1,1) Model**

$$
\sigma_t^2 = \alpha_0 + \alpha_1 X_{t-1}^2 + \beta_1 \sigma_{t-1}^2
$$

This adds **lagged conditional variance** — volatility depends on both the previous shock and the previous volatility level.

- $\alpha_1 + \beta_1 < 1$: stationarity condition
- Typical financial values: $\alpha_1 \approx 0.05\text{–}0.15$, $\beta_1 \approx 0.80\text{–}0.95$
- $\alpha_1 + \beta_1$ close to 1: highly persistent volatility (IGARCH limit)

---

### **6.3 Key Properties**

| Property | ARCH/GARCH |
|---|---|
| $\mathbb{E}[X_t \mid \mathcal{F}_{t-1}]$ | $= 0$ (conditionally centered) |
| $\mathrm{Cov}(X_t, X_{t-k})$ | $= 0$ for $k \geq 1$ (white noise) |
| $\mathrm{Cov}(X_t^2, X_{t-k}^2)$ | $\neq 0$ (squared autocorrelation) |
| Ljung–Box on $X_t$ | Passes (uncorrelated) |
| Ljung–Box on $X_t^2$ | Fails (conditional heteroskedasticity) |
| Distribution | Leptokurtic (heavy tails) even if $z_t$ is Gaussian |

---

### **6.4 Detection and Estimation**

**Detection:**
1. Fit a mean model (e.g., ARMA), extract residuals $\hat{\varepsilon}_t$
2. Compute ACF of $\hat{\varepsilon}_t^2$ — significant structure indicates ARCH effects
3. Engle's ARCH test (Lagrange Multiplier test on $\hat{\varepsilon}_t^2$)

**Estimation:**
- Maximum likelihood (assuming $z_t \sim N(0,1)$ or Student-$t$)
- Packages: `arch` in Python

> **Exam-safe statement:** "GARCH models capture volatility clustering.
> The process is white noise in levels but has autocorrelated conditional variance.
> Ljung–Box on levels passes; on squares it fails."


In [ ]:
# ============================================================
# PART 6: ARCH/GARCH Simulation and Estimation
# ============================================================

np.random.seed(123)
n = 2000

# --- Simulate GARCH(1,1) ---
alpha0, alpha1, beta1 = 0.05, 0.10, 0.85
print(f"GARCH(1,1): alpha0={alpha0}, alpha1={alpha1}, beta1={beta1}")
print(f"Persistence: alpha1 + beta1 = {alpha1 + beta1}")

z = np.random.normal(0, 1, n)
sigma2 = np.zeros(n)
x_garch = np.zeros(n)
sigma2[0] = alpha0 / (1 - alpha1 - beta1)  # unconditional variance

for t in range(1, n):
    sigma2[t] = alpha0 + alpha1 * x_garch[t-1]**2 + beta1 * sigma2[t-1]
    x_garch[t] = np.sqrt(sigma2[t]) * z[t]

# --- Time plots ---
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(x_garch, linewidth=0.5, color="#2c3e50")
axes[0].set_title("GARCH(1,1) Returns (white noise in levels)", fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].plot(np.sqrt(sigma2), linewidth=0.8, color="#e74c3c")
axes[1].set_title("Conditional Volatility $\\sigma_t$ (volatility clustering)", fontsize=12)
axes[1].grid(True, alpha=0.3)

axes[2].plot(x_garch**2, linewidth=0.5, color="#8e44ad")
axes[2].set_title("Squared Returns $X_t^2$ (autocorrelated)", fontsize=12)
axes[2].set_xlabel("$t$")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- ACF of levels vs squares ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
nlags = 30
ci = 1.96 / np.sqrt(n)

a_level = acf(x_garch, nlags=nlags, fft=True)
a_sq = acf(x_garch**2, nlags=nlags, fft=True)

axes[0].stem(range(len(a_level)), a_level, basefmt=" ")
axes[0].axhline(ci, color="red", linestyle="--", alpha=0.5)
axes[0].axhline(-ci, color="red", linestyle="--", alpha=0.5)
axes[0].set_title("ACF of $X_t$ (levels — white noise)", fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].stem(range(len(a_sq)), a_sq, basefmt=" ")
axes[1].axhline(ci, color="red", linestyle="--", alpha=0.5)
axes[1].axhline(-ci, color="red", linestyle="--", alpha=0.5)
axes[1].set_title("ACF of $X_t^2$ (squares — ARCH structure!)", fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Ljung-Box comparison ---
lb_lev = acorr_ljungbox(x_garch, lags=[10, 20], return_df=True)
lb_sq = acorr_ljungbox(x_garch**2, lags=[10, 20], return_df=True)

print("\n=== LJUNG-BOX TESTS ===")
print("\nLevels (should PASS — white noise):")
print(lb_lev.to_string())
print("\nSquares (should FAIL — ARCH effects):")
print(lb_sq.to_string())

# --- Fit GARCH(1,1) using arch package ---
try:
    from arch import arch_model
    am = arch_model(x_garch, vol="Garch", p=1, q=1, mean="Zero")
    res_garch = am.fit(disp="off")
    print("\n=== GARCH(1,1) ESTIMATION (arch package) ===")
    print(res_garch.summary().tables[1])
    print(f"\nTrue:      alpha0={alpha0}, alpha1={alpha1}, beta1={beta1}")
except ImportError:
    print("\n(arch package not installed — run: pip install arch)")
    print("Estimation skipped, but the simulation demonstrates the key concepts.")

print("\nKey takeaways:")
print("  1. Returns are uncorrelated (white noise) — Ljung-Box on levels passes")
print("  2. Squared returns are autocorrelated — Ljung-Box on squares fails")
print("  3. Volatility clusters: high-vol periods followed by high-vol periods")
print("  4. This is the Week 1 ARCH demo revisited, now with GARCH(1,1) and estimation")


## **PART 7 — Synthesis (Exam-Ready)**

> **Core points you must be able to state without hesitation:**

---

### **VAR Models**

- VAR($p$) is the multivariate extension of AR($p$), capturing both own-lag and cross-lag dependence
- Stationarity requires all eigenvalues of the companion matrix inside the unit circle
- Estimation is equation-by-equation OLS; lag order selected by AIC/BIC
- The MA($\infty$) representation yields impulse response functions (IRFs)
- Orthogonalized IRFs (Cholesky) require ordering assumptions about contemporaneous causality

### **Granger Causality**

- $X_2$ Granger-causes $X_1$ if past $X_2$ improves forecasts of $X_1$
- Tested as F-test on cross-lag coefficients in the VAR equation
- Granger causality is **predictive**, not structural or physical
- Requires **stationarity** — testing on $I(1)$ data is invalid
- Direction-dependent and information-set dependent

### **Spurious Regression and Cointegration**

- Regressing $I(1)$ on independent $I(1)$ gives spurious significance (high $R^2$, significant $t$, but nonstationary residuals)
- **Cointegration**: two $I(1)$ series with a stationary linear combination share a long-run equilibrium
- **Engle–Granger**: OLS on levels, then ADF on residuals
- **Johansen**: determines rank (number of cointegrating relationships) in multivariate systems
- **VECM** preserves both short-run dynamics and the long-run equilibrium — differencing alone destroys the equilibrium information

### **ARCH/GARCH**

- GARCH($p,q$) models conditional variance (volatility dynamics)
- Returns are **white noise** (uncorrelated) but **not independent** (squared autocorrelation)
- Ljung–Box on levels: passes. On squares: fails.
- Detection: ACF of squared residuals, Engle's ARCH LM test
- GARCH(1,1): $\sigma_t^2 = \alpha_0 + \alpha_1 X_{t-1}^2 + \beta_1 \sigma_{t-1}^2$
- Stationarity: $\alpha_1 + \beta_1 < 1$; persistence: $\alpha_1 + \beta_1$ close to 1


## **PART 8 — Self-Test: Exam Preparation Questions**

> Work through these without looking at notes first.

---

### **Conceptual Questions (Oral Exam Style)**

**Q1.** Write the VAR(1) model for a bivariate system. What do the off-diagonal elements of $\mathbf{A}$ represent?

**Q2.** What is an impulse response function? How does it relate to the $\psi$-weights from the Wold decomposition?

**Q3.** Why does Cholesky ordering matter for orthogonalized IRFs? Give an example where changing the ordering changes the interpretation.

**Q4.** Define Granger causality precisely. Give three reasons why Granger causality does not imply structural causation.

**Q5.** Explain the spurious regression problem. What diagnostic would you use to detect it?

**Q6.** Two $I(1)$ series, $X_t$ and $Y_t$, have a cointegrating relationship $Y_t - 2X_t \sim I(0)$. Your colleague proposes to model $\Delta Y_t$ and $\Delta X_t$ in a VAR. What is wrong with this approach? What should they use instead?

**Q7.** State the GARCH(1,1) model. Why are GARCH returns white noise but not independent? What specific test reveals the dependence?

**Q8.** Your colleague fits an ARMA model, checks residuals with Ljung–Box (passes), and declares the model adequate. The data is financial returns. What did they miss?

---

### **Technical Questions**

**Q9.** For VAR(1) with $\mathbf{A} = \begin{pmatrix} 0.5 & 0.3 \\ 0.0 & 0.6 \end{pmatrix}$:
- Is the system stable? (Check eigenvalues)
- Does $X_2$ Granger-cause $X_1$? Does $X_1$ Granger-cause $X_2$?
- Compute $\boldsymbol{\Psi}_1 = \mathbf{A}$ and $\boldsymbol{\Psi}_2 = \mathbf{A}^2$

**Q10.** For GARCH(1,1) with $\alpha_0 = 0.01$, $\alpha_1 = 0.08$, $\beta_1 = 0.90$:
- What is the unconditional variance?
- What is the volatility persistence $\alpha_1 + \beta_1$?
- Is this model stationary?

**Q11.** You have two $I(1)$ series. The Engle–Granger residuals have ADF $p = 0.03$. The Johansen test reports rank $r = 1$. What model should you fit?

---

### **Practical Challenge**

**Q12.** Modify the VAR simulation in Part 1 to use $\mathbf{A} = \begin{pmatrix} 0.5 & 0 \\ 0.4 & 0.3 \end{pmatrix}$ (unidirectional: $X_1 \to X_2$ only). Verify with Granger causality tests.

**Q13.** Simulate a GARCH(1,1) with $\alpha_1 + \beta_1 = 0.99$ (near-IGARCH). How does the volatility clustering differ from the $\alpha_1 + \beta_1 = 0.95$ case?
